# ThunderDots — guide utilisateur

Ce notebook présente **ThunderDots** comme une petite bibliothèque Python pour interroger un endpoint **DTS** (*Distributed Text Services*), parcourir des collections, récupérer des ressources textuelles, extraire des fragments, filtrer les métadonnées, valider les sorties et produire des objets exploitables dans une chaîne d’indexation.

L’objectif est de montrer les usages du plus simple au plus avancé :

1. récupérer une collection DTS ;
2. inspecter les ressources et les fragments ;
3. choisir les métadonnées à conserver ;
4. comprendre les modes de fragmentation ;
5. comparer un découpage basé sur la navigation DTS et un découpage non basé sur la navigation ;
6. valider les résultats ;
7. exporter vers des formats pratiques pour Elasticsearch, Qdrant ou une chaîne de traitement personnalisée.

> Le notebook reste volontairement générique. Les exemples utilisent des identifiants et URLs fictifs ou paramétrables. Remplacez-les par votre propre endpoint DTS et vos propres collections.

## 1. Installation et imports

ThunderDots dépend principalement de `httpx`, `lxml`, `rich` et `jsonschema`. Si la bibliothèque est installée en mode développement dans votre projet, l’import suivant suffit.

```bash
pip install -e .
```

Dans un notebook lancé depuis la racine du dépôt, on peut aussi ajouter `src/` au `PYTHONPATH`.

In [ ]:
import pprint

# À activer si vous exécutez ce notebook depuis la racine du dépôt :
# import sys
# sys.path.insert(0, str(Path.cwd() / "src"))

from thunderdots import ThunderDots
from thunderdots.validation import validate_notice, validate_many

pp = pprint.PrettyPrinter(width=120, sort_dicts=False)

## 2. Paramètres de base

Un endpoint DTS expose généralement plusieurs routes, dont :

- `/collection` pour parcourir des collections et sous-collections ;
- `/document` pour récupérer le texte ou XML d’une ressource ;
- `/navigation` pour récupérer une table de navigation ou structure citationnelle quand elle existe.

Dans ThunderDots, on fournit seulement la racine de l’API DTS. Les chemins `/collection`, `/document` et `/navigation` sont construits ensuite par la bibliothèque.

In [2]:
ENDPOINT_DTS = "https://dev.chartes.psl.eu/dots/api/dts"  # À remplacer
COLLECTION_ID = "ENCPOS_1972"  # À remplacer, ou None pour partir de la racine

OUTPUT_JSON = "./artifacts/thunderdots/results.json"
CACHE_CSV = "./artifacts/thunderdots/resources_cache.csv"

## 3. Premier usage : récupérer une collection et ses ressources

Le cas minimal consiste à fournir :

- `endpoint_dts` : l’endpoint DTS ;
- `collection_params.collection_id` : la collection de départ ;
- `verbose=True` pour afficher une barre de progression.

Par défaut, ThunderDots :

- parcourt les collections à partir de `collection_id` ;
- récupère les ressources rencontrées ;
- récupère le document XML ou texte ;
- utilise la navigation DTS si elle est disponible ;
- retourne un dictionnaire avec `collection_results`, `resource_results` et `meta`.

In [4]:
td = ThunderDots(
    endpoint_dts=ENDPOINT_DTS,
    collection_params={
        "collection_id": COLLECTION_ID,
    },
    verbose=True,
    use_cache=False,
)

# Décommentez pour exécuter l'appel réseau.
td.fetch()
results = td.results()
pp.pprint(results.keys())

⚡ ThunderDots ✔ Done  elapsed=6.12s  http_errors=0

dict_keys(['dtsVersion', 'type', 'meta', 'collection_results', 'resource_results'])


La sortie générale ressemble à ceci :

```python
{
    "dtsVersion": "1-alpha",
    "type": "All",
    "meta": {...},
    "collection_results": [...],
    "resource_results": [...]
}
```

Chaque entrée de `resource_results` représente une ressource DTS enrichie par ThunderDots :

```python
{
    "id": "resource_id",
    "@type": "Resource",
    "title": "Titre de la ressource",
    "linked_parents": [...],
    "metadata": {...},
    "fragments": [...]
}
```

## 4. Inspection rapide des résultats

Les méthodes `results()`, `collection_results()`, `resource_results()` et `stats()` donnent des vues pratiques sur les données récupérées.

In [5]:
# Après td.fetch() :
all_results = td.results()
collections_only = td.collection_results()
resources_only = td.resource_results()
stats = td.stats()

# Exemple d'affichage :
print("Collections:", len(all_results.get("collection_results", [])))
print("Ressources:", len(all_results.get("resource_results", [])))
pp.pprint(stats)

Collections: 1
Ressources: 24
{'timestamp': '2026-05-13T12:54:46.274246', 'elapsed_seconds': 6.116166830062866, 'http_errors': 0}


In [15]:
def summarize_results(results: dict, preview_chars: int = 300) -> None:
    resources = results.get("resource_results", [])
    print(f"Ressources : {len(resources)}")

    for resource in resources[:10]:
        fragments = resource.get("fragments", [])
        print("=" * 80)
        print("ID        :", resource.get("id"))
        print("Titre     :", resource.get("title"))
        print("Fragments :", len(fragments))
        if fragments:
            first = fragments[0]
            print("dots_id   :", first.get("dots_id"))
            print("head      :", first.get("head"))
            content = " ".join((first.get("content") or "").split())
            print(
                "Aperçu    :",
                content[:preview_chars] + ("…" if len(content) > preview_chars else ""),
            )


summarize_results(results)

Ressources : 24
ID        : ENCPOS_1972_02
Titre     : Le prieuré de Saint-Martin-des-Champs à Paris : étude historique et archéologique
Fragments : 16
dots_id   : r951105
head      : Sources
Aperçu    : Sources Le fonds du prieuré de Saint-Martin-des-Champs à Paris se trouve essentiellement aux Archives nationales, où il est réparti entre les séries H5 (comptes du xviiie siècle), L, LL (cartulaires et censiers, livres de visites, registres capitulaires et comptes allant du xiiie siècle à la Révolut…
ID        : ENCPOS_1972_01
Titre     : Les statuts d’une colonie génoise en Corse : Bonifacio à la fin du Moyen Âge
Fragments : 8
dots_id   : r950666
head      : Introduction
Aperçu    : Introduction Pour se maintenir dans l’île de Corse, enjeu principal de l’antagonisme politique et commercial qui opposait en Méditerranée occidentale Pisans et Génois, la République de Gênes avait fondé deux villes-forteresses sur des lieux qu’elle considérait comme les positions clés de l’île : Bon…
ID   

## 5. Choisir les métadonnées à conserver

ThunderDots distingue les métadonnées de collection et les métadonnées de ressource.

Les paramètres recommandés sont :

- `metadata_dublincore` pour conserver des champs Dublin Core ;
- `metadata_extensions` pour conserver des champs d’extension.

L’ancien paramètre `keep_metadata` reste accepté par compatibilité, mais il est préférable d’utiliser les deux paramètres explicites.

In [10]:
td = ThunderDots(
    endpoint_dts=ENDPOINT_DTS,
    collection_params={
        "collection_id": COLLECTION_ID,
        "metadata_dublincore": ["title"],
    },
    resource_params={
        "metadata_dublincore": ["identifier", "title", "creator", "date", "coverage"],
        "metadata_extensions": ["dct:coverage"],
    },
    verbose=True,
    use_cache=False,
)

td.fetch()
results = td.results()

⚡ ThunderDots ✔ Done  elapsed=5.20s  http_errors=0

### À quoi sert ce filtrage ?

Le filtrage évite de transporter toutes les métadonnées DTS brutes quand on veut produire un résultat compact et stable. Il est utile pour :

- alimenter un index documentaire ;
- limiter la taille du JSON final ;
- éviter de dépendre de champs instables ;
- construire ensuite des filtres propres dans Elasticsearch, Qdrant ou un moteur maison.

## 6. Comprendre les fragments

ThunderDots produit une liste de fragments pour chaque ressource. Un fragment est l’unité documentaire minimale que vous pouvez ensuite envoyer dans une chaîne d’indexation, de recherche ou de traitement.

Un fragment contient au minimum :

```python
{
    "dots_id": "...",
    "content": "..."
}
```

Selon le mode utilisé, il peut aussi contenir :

```python
{
    "head": "Titre de section",
    "breadcrumb": "Chemin > Vers > Section",
    "level": 1,
    "fragment_xpath": ".//tei:text/tei:body/tei:div",
    "fragment_index": 0
}
```

Le champ `dots_id` est important : c’est l’identifiant stable du fragment. Il peut venir de la navigation DTS, d’un `xml:id` TEI, ou être généré par hash SHA1 quand aucun identifiant local n’existe.

## 7. Les modes de fragmentation

ThunderDots supporte trois modes explicites, plus un mode automatique.

| Mode | Source de structure | Usage typique |
|---|---|---|
| `auto` | navigation si disponible, sinon document global | choix par défaut |
| `navigation` | `/navigation` + `/document` | utiliser la structure déclarée par le serveur DTS |
| `document` | `/document` seulement | un seul fragment par ressource |
| `tei_xpath` | `/document` seulement + XPath TEI | découpage personnalisé non dépendant de `/navigation` |

Le mode choisi se configure dans `resource_params.fragment_mode`.

## 8. Mode `document` : un fragment global par ressource

Ce mode est le plus simple pour récupérer le texte complet d’une ressource. Il n’utilise pas la navigation DTS.

Il produit généralement un seul fragment :

```python
{
    "dots_id": "__DOCUMENT__",
    "content": "texte complet de la ressource"
}
```

Si `add_head_to_content=False`, les balises TEI `<head>` sont retirées du contenu. C’est utile lorsque les titres ne doivent pas polluer le texte plein, par exemple avant un découpage secondaire dans un pipeline de chunks.

In [11]:
td_document = ThunderDots(
    endpoint_dts=ENDPOINT_DTS,
    collection_params={
        "collection_id": COLLECTION_ID,
    },
    resource_params={
        "fragment_mode": "document",
        "fetch_document": True,
        "fetch_navigation": False,
        "add_head_to_content": False,
        "include_breadcrumb": False,
    },
    verbose=True,
    use_cache=False,
)

td_document.fetch()
document_results = td_document.results()
summarize_results(document_results)

⚡ ThunderDots ✔ Done  elapsed=2.14s  http_errors=0

Ressources : 24
ID        : ENCPOS_1972_01
Titre     : Les statuts d’une colonie génoise en Corse : Bonifacio à la fin du Moyen Âge
Fragments : 1
dots_id   : __DOCUMENT__
head      : None
Aperçu    : par Marie-Claude Bartoli maître ès lettres Pour se maintenir dans l’île de Corse, enjeu principal de l’antagonisme politique et commercial qui opposait en Méditerranée occidentale Pisans et Génois, la République de Gênes avait fondé deux villes-forteresses sur des lieux qu’elle considérait comme les…
ID        : ENCPOS_1972_14
Titre     : Marle à la fin de l’Ancien Régime : étude de la société et de la vie sociale dans une petite ville de 1760 à 1789
Fragments : 1
dots_id   : __DOCUMENT__
head      : None
Aperçu    : par Dominique Morelon-Péry Aux Archives départementales de l’Aisne nous avons examiné une partie des registres de catholicité, les minutes notariales provenant de deux des quatre études qui étaient alors installées à Marle, et une partie du fonds du bailliage. Différents artic

### Quand utiliser `document` ?

Utilisez `document` si :

- vous voulez un document complet par ressource ;
- vous prévoyez de faire ensuite votre propre chunking ;
- la navigation DTS est absente, incomplète ou trop fine ;
- votre moteur aval attend une seule notice textuelle par ressource.

Limite : le fragment peut devenir très long. Pour une base vectorielle, il faudra souvent effectuer un chunking ensuite.

## 9. Mode `navigation` : fragments fondés sur la navigation DTS

Le mode `navigation` utilise `/navigation` pour connaître les unités de citation ou sections déclarées par le serveur, puis récupère le XML avec `/document`.

ThunderDots associe ensuite les entrées de navigation aux nœuds TEI via leurs identifiants. Cela permet de produire des fragments structurés avec `dots_id`, `head`, `level` et `breadcrumb`.

In [16]:
td_navigation = ThunderDots(
    endpoint_dts=ENDPOINT_DTS,
    collection_params={
        "collection_id": COLLECTION_ID,
    },
    resource_params={
        "fragment_mode": "navigation",
        "fetch_document": True,
        "fetch_navigation": True,
        "add_head_to_content": False,
        "include_breadcrumb": True,
        "exclude_heads_contains": [
            "index",
            "appendices",
            "annexes",
            "sources",
            "bibliographie",
            "iconographie",
        ],
    },
    verbose=True,
    use_cache=False,
)

td_navigation.fetch()
navigation_results = td_navigation.results()
summarize_results(navigation_results)

⚡ ThunderDots ✔ Done  elapsed=5.32s  http_errors=0

Ressources : 24
ID        : ENCPOS_1972_08
Titre     : Les routes de la généralité de Châlons-sur-Marne au xviiie siècle
Fragments : 13
dots_id   : r954768
head      : Introduction
Aperçu    : Le dix-huitième siècle a été véritablement le « grand siècle » des routes. Au cours de cette période s’est élaboré un système organisé des voies de communications terrestres qui s’est maintenu jusqu’à nos jours et sur lequel le réseau routier actuel s’est modelé. Cette réalisation a été rendue possi…
ID        : ENCPOS_1972_02
Titre     : Le prieuré de Saint-Martin-des-Champs à Paris : étude historique et archéologique
Fragments : 14
dots_id   : r951205
head      : Introduction les origines du prieuré
Aperçu    : En 1060, le roi Henri Ier fonda le monastère et y installa des chanoines réguliers. Les termes mêmes de la charte royale laissent entendre qu’il existait antérieurement une abbaye de Saint-Martin et qu’elle fut détruite de fond en comble par les Normands. Un diplôme de Childebert III, du

### Quand utiliser `navigation` ?

Utilisez `navigation` si :

- le serveur DTS expose une navigation fiable ;
- vous voulez respecter la structure éditoriale publiée ;
- les identifiants de fragments doivent correspondre aux identifiants citables du serveur ;
- vous voulez des breadcrumbs exploitables en interface utilisateur.

Limite : vous dépendez de ce que le serveur déclare dans `/navigation`. Si la navigation est absente, trop grossière ou trop fine, préférez `tei_xpath`.

## 10. Mode `tei_xpath` : fragmentation personnalisée sans navigation

Le mode `tei_xpath` récupère uniquement `/document`, puis découpe le XML TEI avec un XPath fourni par l’utilisateur.

C’est le mode à privilégier quand vous voulez contrôler vous-même la granularité documentaire, par exemple :

- un fragment par `<div>` ;
- un fragment par `<p>` ;
- un fragment par `<ab>` ;
- un fragment par nœud métier propre à votre schéma XML.

Ce mode est **non basé sur la navigation DTS**.

In [18]:
td_xpath_div = ThunderDots(
    endpoint_dts=ENDPOINT_DTS,
    collection_params={
        "collection_id": COLLECTION_ID,
    },
    resource_params={
        "fragment_mode": "tei_xpath",
        "fragment_xpath": ".//tei:text/tei:body/tei:div",
        "title_xpath": "./tei:head",
        "remove_fragment_heads": True,
        "add_head_to_content": False,
        "generated_id_prefix": "__DOCUMENT__",
        "fetch_document": True,
        "fetch_navigation": False,
        "include_breadcrumb": True,
        "exclude_heads_contains": [
            "index",
            "appendices",
            "annexes",
            "sources",
            "bibliographie",
            "iconographie",
        ],
    },
    verbose=True,
    use_cache=False,
)

td_xpath_div.fetch()
xpath_div_results = td_xpath_div.results()
summarize_results(xpath_div_results)

⚡ ThunderDots ✔ Done  elapsed=2.64s  http_errors=0

Ressources : 24
ID        : ENCPOS_1972_06
Titre     : Les plus anciennes chartes en langue française de l’Aube et de la Seine-et-Marne
Fragments : 4
dots_id   : r953815
head      : Introduction
Aperçu    : La connaissance de la langue médiévale, et en particulier des dialectes régionaux, s’appuie de plus en plus sur l’étude des documents d’archives, qui ont l’avantage sur les textes littéraires de pouvoir être localisés et datés de manière beaucoup plus précise. C’est dans le cadre de la publication s…
ID        : ENCPOS_1972_20
Titre     : Guillaume de Varye, facteur de Jacques Cœur et général des finances de Louis XI
Fragments : 6
dots_id   : r961696
head      : Introduction
Aperçu    : A Bourges, où Guillaume de Varye est né, une rue porte son nom et les plus vieux habitants de la ville peuvent se rappeler un hôtel de belle allure, maintenant démoli, qui avait été construit sur la boutique de ses ancêtres. C’est là que Guillaume de Varye, devenu l’ami et le collaborateur, sinon le…

### Exemple : un fragment par paragraphe TEI

Pour une granularité plus fine, on peut choisir les paragraphes :

```python
"fragment_xpath": ".//tei:text/tei:body/tei:div/tei:p"
```

Dans ce cas, les `<p>` n’ont pas toujours de `<head>` local. ThunderDots remonte alors vers le premier ancêtre ayant un `<head>` grâce à la logique interne de `_nearest_ancestor_head`. Le titre de section est donc conservé dans `head`, mais retiré du contenu si `add_head_to_content=False`.

In [19]:
td_xpath_p = ThunderDots(
    endpoint_dts=ENDPOINT_DTS,
    collection_params={
        "collection_id": COLLECTION_ID,
    },
    resource_params={
        "fragment_mode": "tei_xpath",
        "fragment_xpath": ".//tei:text/tei:body/tei:div/tei:p",
        "title_xpath": "./tei:head",
        "remove_fragment_heads": True,
        "add_head_to_content": False,
        "generated_id_prefix": "__DOCUMENT__",
        "fetch_document": True,
        "fetch_navigation": False,
        "include_breadcrumb": True,
    },
    verbose=True,
    use_cache=False,
)

td_xpath_p.fetch()
xpath_p_results = td_xpath_p.results()
summarize_results(xpath_p_results)

⚡ ThunderDots ✔ Done  elapsed=2.05s  http_errors=0

Ressources : 24
ID        : ENCPOS_1972_02
Titre     : Le prieuré de Saint-Martin-des-Champs à Paris : étude historique et archéologique
Fragments : 12
dots_id   : __DOCUMENT__e1bff1e833f4
head      : Sources
Aperçu    : Le fonds du prieuré de Saint-Martin-des-Champs à Paris se trouve essentiellement aux Archives nationales, où il est réparti entre les séries H5 (comptes du xviiie siècle), L, LL (cartulaires et censiers, livres de visites, registres capitulaires et comptes allant du xiiie siècle à la Révolution) et …
ID        : ENCPOS_1972_01
Titre     : Les statuts d’une colonie génoise en Corse : Bonifacio à la fin du Moyen Âge
Fragments : 24
dots_id   : __DOCUMENT__5d5ab6c7aec4
head      : Introduction
Aperçu    : Pour se maintenir dans l’île de Corse, enjeu principal de l’antagonisme politique et commercial qui opposait en Méditerranée occidentale Pisans et Génois, la République de Gênes avait fondé deux villes-forteresses sur des lieux qu’elle considérait comme les positions clés

## 11. Différence entre `navigation` et `tei_xpath`

Les deux modes peuvent produire plusieurs fragments par ressource, mais ils ne répondent pas au même besoin.

### `navigation`

- La granularité vient du serveur DTS.
- Les `dots_id` correspondent en général aux identifiants exposés dans `/navigation`.
- Les breadcrumbs reflètent la hiérarchie DTS.
- C’est le bon choix si l’on veut rester aligné avec le modèle de citation publié.

### `tei_xpath`

- La granularité vient de votre expression XPath.
- Les `dots_id` viennent du `xml:id` du nœud si disponible.
- Si le nœud n’a pas de `xml:id`, ThunderDots génère un identifiant stable avec un préfixe et un hash SHA1.
- C’est le bon choix si vous voulez définir votre propre unité documentaire : paragraphe, division, bloc, item, etc.

En résumé :

```text
navigation = je fais confiance à la structure DTS publiée
tei_xpath  = je définis moi-même la granularité dans le XML TEI
```

In [22]:
from typing import Any


def compare_fragmentation(*named_results: tuple[str, dict[str, Any]]) -> None:
    """
    Compare le nombre de ressources et de fragments pour plusieurs résultats ThunderDots.

    Usage
    -----
    compare_fragmentation(
        ("navigation", navigation_results),
        ("tei_xpath div", xpath_div_results),
        ("tei_xpath p", xpath_p_results),
    )
    """

    def count_fragments(results: dict[str, Any]) -> tuple[int, int]:
        resources = results.get("resource_results", [])
        fragments = sum(len(resource.get("fragments", [])) for resource in resources)
        return len(resources), fragments

    if not named_results:
        print("Aucun résultat à comparer.")
        return

    for label, results in named_results:
        resources_count, fragments_count = count_fragments(results)
        print(f"{label}: {resources_count} ressources, {fragments_count} fragments")


compare_fragmentation(
    ("navigation", navigation_results),
    ("tei_xpath div", xpath_div_results),
    ("tei_xpath p", xpath_p_results),
)

navigation: 24 ressources, 310 fragments
tei_xpath div: 24 ressources, 123 fragments
tei_xpath p: 24 ressources, 259 fragments


## 12. Exclure des sections par titre

Le paramètre `exclude_heads_contains` permet d’exclure des fragments dont le titre contient certains mots ou expressions.

La comparaison est :

- insensible à la casse ;
- insensible aux accents ;
- basée sur une inclusion simple.

Exemple : si un fragment a pour titre `Pièces annexes`, il sera exclu par le motif `annexes` ou `pièces annexes`.

In [23]:
COMMON_EXCLUDED_HEADS = [
    "index",
    "appendices",
    "annexes",
    "sources",
    "bibliographie",
    "iconographie",
    "lexique",
    "cartes et plans",
    "pièces justificatives",
]

## 13. Gestion des titres dans le contenu

Deux paramètres contrôlent la présence des titres dans les fragments :

- `add_head_to_content` ;
- `remove_fragment_heads`.

### `add_head_to_content=True`

Le titre est ajouté au début du contenu. C’est utile pour des systèmes de recherche où le contexte du titre doit contribuer au score.

### `add_head_to_content=False`

Le titre reste disponible dans le champ `head`, mais il n’est pas injecté dans `content`. C’est utile quand on veut éviter que les titres dominent le texte indexé.

### `remove_fragment_heads=True`

En mode `tei_xpath`, les `<head>` locaux sont retirés du texte extrait du nœud, afin d’éviter les doublons.

In [26]:
td_titles_in_content = ThunderDots(
    endpoint_dts=ENDPOINT_DTS,
    collection_params={"collection_id": COLLECTION_ID},
    resource_params={
        "fragment_mode": "tei_xpath",
        "fragment_xpath": ".//tei:text/tei:body/tei:div",
        "title_xpath": "./tei:head",
        "remove_fragment_heads": True,
        "add_head_to_content": True,
    },
    verbose=False,
    use_cache=False,
)

td_titles_in_content.fetch()
td_titles_in_content_results = td_titles_in_content.results()
summarize_results(td_titles_in_content_results)

Ressources : 24
ID        : ENCPOS_1972_02
Titre     : Le prieuré de Saint-Martin-des-Champs à Paris : étude historique et archéologique
Fragments : 8
dots_id   : r951105
head      : Sources
Aperçu    : Sources Le fonds du prieuré de Saint-Martin-des-Champs à Paris se trouve essentiellement aux Archives nationales, où il est réparti entre les séries H5 (comptes du xviiie siècle), L, LL (cartulaires et censiers, livres de visites, registres capitulaires et comptes allant du xiiie siècle à la Révolut…
ID        : ENCPOS_1972_06
Titre     : Les plus anciennes chartes en langue française de l’Aube et de la Seine-et-Marne
Fragments : 7
dots_id   : r953815
head      : Introduction
Aperçu    : Introduction La connaissance de la langue médiévale, et en particulier des dialectes régionaux, s’appuie de plus en plus sur l’étude des documents d’archives, qui ont l’avantage sur les textes littéraires de pouvoir être localisés et datés de manière beaucoup plus précise. C’est dans le cadre de la …
ID

## 14. Cache JSON et cache CSV

ThunderDots peut écrire deux artefacts :

- `output_path` : le résultat complet JSON ;
- `cache_csv_path` : une table CSV aplatie avec les ressources, le nombre de fragments, la longueur textuelle et les métadonnées conservées.

Si `use_cache=True` et que `output_path` existe déjà, ThunderDots recharge le JSON au lieu de refaire les appels réseau.

In [30]:
td_cached = ThunderDots(
    endpoint_dts=ENDPOINT_DTS,
    collection_params={"collection_id": COLLECTION_ID},
    resource_params={
        "fragment_mode": "auto",
        "metadata_dublincore": ["identifier", "title", "creator", "date"],
    },
    output_path=OUTPUT_JSON,
    cache_csv_path=CACHE_CSV,
    use_cache=True,
    verbose=True,
)

# Premier appel : écrit le cache si OUTPUT_JSON n'existe pas.
# Appels suivants : recharge OUTPUT_JSON si use_cache=True.
td_cached.fetch()

## 15. Validation des résultats

ThunderDots peut valider automatiquement la forme du JSON produit avec `jsonschema`.

Deux niveaux sont particulièrement utiles :

- validation de la sortie complète (`profile="output"`) ;
- validation de chaque ressource produite (`profile="resource_result"`).

Quand `validate=True`, ThunderDots ajoute une clé `validation` dans les résultats.

In [31]:
td_validated = ThunderDots(
    endpoint_dts=ENDPOINT_DTS,
    collection_params={"collection_id": COLLECTION_ID},
    resource_params={
        "fragment_mode": "auto",
    },
    validate=True,
    verbose=True,
    use_cache=False,
)

td_validated.fetch()
validated_results = td_validated.results()
pp.pprint(validated_results.get("validation"))

⚡ ThunderDots ✔ Done  elapsed=5.46s  http_errors=0

{'output': {'ok': True, 'triple_count': None, 'issues': []},
 'resources': {'total': 24, 'valid': 24, 'invalid': 0, 'issues': 0}}


### Comment lire le rapport de validation ?

Un rapport de validation contient :

```python
{
    "ok": True,
    "triple_count": None,
    "issues": []
}
```

- `ok=True` signifie que l’objet respecte le schéma attendu ;
- `ok=False` signifie qu’au moins une erreur a été détectée ;
- `issues` liste les erreurs avec un chemin, un message, la valeur attendue et la valeur observée.

Pour un lot de ressources, le résumé ressemble à ceci :

```python
{
    "total": 100,
    "valid": 98,
    "invalid": 2,
    "issues": 3
}
```

La validation ne garantit pas que le contenu scientifique ou documentaire est correct ; elle garantit que la structure JSON est exploitable par la suite.

In [33]:
# Validation manuelle d'une sortie complète :
output_report = validate_notice(results, profile="output")
pp.pprint(output_report.to_dict())

# Validation manuelle des ressources :
resource_report = validate_many(results.get("resource_results", []), profile="resource_result")
pp.pprint(resource_report.summary())

{'ok': True, 'triple_count': None, 'issues': []}
{'total': 24, 'valid': 24, 'invalid': 0, 'issues': 0}


## 16. Utiliser l’API asynchrone dans un notebook

ThunderDots expose aussi `afetch()`. Elle est utile dans les environnements déjà asynchrones.

Dans un notebook Jupyter moderne, on peut utiliser `await` directement au niveau d’une cellule.

In [34]:
td_async = ThunderDots(
    endpoint_dts=ENDPOINT_DTS,
    collection_params={"collection_id": COLLECTION_ID},
    resource_params={"fragment_mode": "document"},
    verbose=True,
    use_cache=False,
)

await td_async.afetch()
async_results = td_async.results()

⚡ ThunderDots ✔ Done  elapsed=2.37s  http_errors=0

## 17. Transformer les résultats en notices Python

La méthode `notices()` convertit les résultats en objets `DotsNotice`. Ces objets exposent des méthodes pratiques pour produire des documents d’indexation.

In [35]:
# Après td.fetch() :
notices = td.notices()
first = notices[0]
print(first.id)
print(first.title)
print(first.full_text[:500])
pp.pprint(first.creator_names)
pp.pprint(first.temporal_index)

ENCPOS_1972_02
Le prieuré de Saint-Martin-des-Champs à Paris : étude historique et archéologique
Sources Le fonds du prieuré de Saint-Martin-des-Champs à Paris se trouve essentiellement aux Archives nationales, où il est réparti entre les séries H5 (comptes du xviiie siècle), L, LL (cartulaires et censiers, livres de visites, registres capitulaires et comptes allant du xiiie siècle à la Révolution) et S, qui présentent des documents très variés aussi bien sur le prieuré même que sur ses nombreuses dépendances. Les actes concernant Saint-Martin-des-Champs jusqu’à la fin du xiiie siècle ont é
['Catherine Berthier-Georgesco']
{'dublincore.identifier': 'https://dev.chartes.psl.eu/dots/api/dts/collection?id=ENCPOS_1972_02',
 'dublincore.title': 'Le prieuré de Saint-Martin-des-Champs à Paris : étude historique et archéologique',
 'dublincore.creator': 'Catherine Berthier-Georgesco',
 'dublincore.coverage': '0500/1972',
 'dublincore.coverage_start': 500,
 'dublincore.coverage_end': 1972}


## 18. Exporter vers Elasticsearch

ThunderDots peut produire des documents ou des actions compatibles avec les conventions habituelles d’Elasticsearch.

- `to_elastic_documents()` produit une liste de dictionnaires ;
- `to_elastic_actions(index="...")` produit une liste d’actions avec `_index`, `_id` et `_source`.

In [36]:
# Après td.fetch() :
elastic_docs = td.to_elastic_documents(include_fragments=True, include_raw=False)
elastic_actions = td.to_elastic_actions(index="my_index", include_fragments=True, include_raw=False)

pp.pprint(elastic_docs[0])
pp.pprint(elastic_actions[0])

{'id': 'ENCPOS_1972_02',
 'type': 'Resource',
 'title': 'Le prieuré de Saint-Martin-des-Champs à Paris : étude historique et archéologique',
 'text': 'Sources Le fonds du prieuré de Saint-Martin-des-Champs à Paris se trouve essentiellement aux Archives '
         'nationales, où il est réparti entre les séries H5 (comptes du xviiie siècle), L, LL (cartulaires et '
         'censiers, livres de visites, registres capitulaires et comptes allant du xiiie siècle à la Révolution) et S, '
         'qui présentent des documents très variés aussi bien sur le prieuré même que sur ses nombreuses dépendances. '
         'Les actes concernant Saint-Martin-des-Champs jusqu’à la fin du xiiie siècle ont été publiés par J. Depoin '
         'entre 1912 et 1921. Des manuscrits conservés à la Bibliothèque Nationale (notamment nouv. acq. lat. 1 359 et '
         'lat. 10 977, Liber Testamentorum) et à la Bibliothèque Mazarine complètent ces sources, ainsi que les '
         'ouvrages d’érudition du premi

## 19. Exporter vers Qdrant ou une base vectorielle

ThunderDots peut produire des payloads ou points Qdrant.

- `to_qdrant_payloads()` produit seulement les payloads ;
- `to_qdrant_points(vectors=...)` produit les points avec payload et vecteur.

La génération de vecteurs n’est pas faite par ThunderDots : elle dépend de votre modèle d’embedding ou de votre chaîne d’indexation.

In [37]:
# Après td.fetch() :
payloads = td.to_qdrant_payloads(include_fragments=True, include_raw=False)

# Exemple avec vecteurs factices, uniquement pour montrer la forme attendue :
vectors = [[0.0] * 384 for _ in payloads]
points = td.to_qdrant_points(vectors=vectors, include_fragments=True, include_raw=False)
pp.pprint(points[0])

{'id': 5053030632820039882,
 'payload': {'id': 'ENCPOS_1972_02',
             'record_id': 'ENCPOS_1972_02',
             'type': 'Resource',
             'title': 'Le prieuré de Saint-Martin-des-Champs à Paris : étude historique et archéologique',
             'text': 'Sources Le fonds du prieuré de Saint-Martin-des-Champs à Paris se trouve essentiellement aux '
                     'Archives nationales, où il est réparti entre les séries H5 (comptes du xviiie siècle), L, LL '
                     '(cartulaires et censiers, livres de visites, registres capitulaires et comptes allant du xiiie '
                     'siècle à la Révolution) et S, qui présentent des documents très variés aussi bien sur le prieuré '
                     'même que sur ses nombreuses dépendances. Les actes concernant Saint-Martin-des-Champs jusqu’à la '
                     'fin du xiiie siècle ont été publiés par J. Depoin entre 1912 et 1921. Des manuscrits conservés à '
                     'la Bibliothèq

## 20. Construire ses propres enregistrements d’indexation

Dans beaucoup de cas, on veut transformer les fragments ThunderDots en documents plats, par exemple pour une chaîne de recherche plein texte, une base vectorielle, un pipeline RAG ou un traitement linguistique.

La fonction suivante produit un document par fragment.

In [38]:
def iter_fragment_documents(results: dict):
    for resource in results.get("resource_results", []):
        resource_id = resource.get("id")
        title = resource.get("title")
        metadata = resource.get("metadata") or {}
        linked_parents = resource.get("linked_parents") or []

        for index, fragment in enumerate(resource.get("fragments", [])):
            content = (fragment.get("content") or "").strip()
            if not content:
                continue

            yield {
                "id": f"{resource_id}__frag_{index}",
                "record_id": resource_id,
                "dots_id": fragment.get("dots_id"),
                "title": title,
                "head": fragment.get("head"),
                "breadcrumb": fragment.get("breadcrumb"),
                "text": content,
                "metadata": metadata,
                "linked_parents": linked_parents,
            }


docs = list(iter_fragment_documents(results))
pp.pprint(docs[0])

{'id': 'ENCPOS_1972_02__frag_0',
 'record_id': 'ENCPOS_1972_02',
 'dots_id': 'r951105',
 'title': 'Le prieuré de Saint-Martin-des-Champs à Paris : étude historique et archéologique',
 'head': 'Sources',
 'breadcrumb': 'Sources',
 'text': 'Sources Le fonds du prieuré de Saint-Martin-des-Champs à Paris se trouve essentiellement aux Archives '
         'nationales, où il est réparti entre les séries H5 (comptes du xviiie siècle), L, LL (cartulaires et '
         'censiers, livres de visites, registres capitulaires et comptes allant du xiiie siècle à la Révolution) et S, '
         'qui présentent des documents très variés aussi bien sur le prieuré même que sur ses nombreuses dépendances. '
         'Les actes concernant Saint-Martin-des-Champs jusqu’à la fin du xiiie siècle ont été publiés par J. Depoin '
         'entre 1912 et 1921. Des manuscrits conservés à la Bibliothèque Nationale (notamment nouv. acq. lat. 1 359 et '
         'lat. 10 977, Liber Testamentorum) et à la Bibliothèque 

## 21. Paramètres principaux de `ThunderDots`

| Paramètre | Type | Rôle |
|---|---:|---|
| `endpoint_dts` | `str` | URL racine de l’API DTS |
| `collection_params` | `dict` | paramètres de parcours des collections |
| `resource_params` | `dict` | paramètres de récupération et fragmentation des ressources |
| `validate` | `bool` | ajoute un rapport de validation JSON |
| `verbose` | `bool` | active l’interface de progression Rich |
| `concurrency` | `int` | nombre de workers concurrents |
| `request_timeout` | `float` | timeout HTTP par requête |
| `retries` | `int` | nombre de tentatives en cas d’échec temporaire |
| `backoff_ms` | `int` | délai de backoff entre les tentatives |
| `output_path` | `str | None` | chemin du JSON complet |
| `cache_csv_path` | `str | None` | chemin du CSV de synthèse |
| `use_cache` | `bool` | recharge `output_path` s’il existe |

## 22. Paramètres de collection

`collection_params` contrôle le point de départ et le filtrage des collections.

| Paramètre | Type | Rôle |
|---|---:|---|
| `collection_id` | `str | None` | collection de départ ; `None` part de la racine |
| `excluded_ids` | `list[str]` | collections ou ressources à ignorer lors du parcours |
| `metadata_dublincore` | `list[str]` | champs Dublin Core à conserver pour les collections |
| `metadata_extensions` | `list[str]` | champs d’extension à conserver pour les collections |

Exemple :

In [ ]:
collection_params = {
    "collection_id": COLLECTION_ID,
    "excluded_ids": ["collection_to_skip"],
    "metadata_dublincore": ["title"],
    "metadata_extensions": [],
}

## 23. Paramètres de ressource

`resource_params` contrôle la récupération des documents, la navigation et la fragmentation.

| Paramètre | Type | Défaut | Rôle |
|---|---:|---:|---|
| `metadata_dublincore` | `list[str]` | `[]` | champs Dublin Core de ressource |
| `metadata_extensions` | `list[str]` | `[]` | champs d’extension de ressource |
| `add_head_to_content` | `bool` | `True` | ajoute les titres dans le texte |
| `include_breadcrumb` | `bool` | `True` | ajoute un fil d’Ariane si disponible |
| `exclude_heads_contains` | `list[str]` | `[]` | exclut des fragments par titre |
| `fetch_document` | `bool` | `True` | récupère `/document` |
| `fetch_navigation` | `bool` | `True` | récupère `/navigation` si utile |
| `fragment_mode` | `str` | `auto` | `auto`, `navigation`, `document`, `tei_xpath` |
| `fragment_xpath` | `str | None` | `None` | XPath TEI des fragments en mode `tei_xpath` |
| `title_xpath` | `str` | `./tei:head` | XPath du titre local |
| `remove_fragment_heads` | `bool` | `True` | retire les `<head>` locaux du contenu |
| `generated_id_prefix` | `str` | `__DOCUMENT__` | préfixe des IDs générés |

## 24. Modèles de configuration prêts à copier

### A. Récupération simple, document global

In [ ]:
config_document_global = {
    "collection_params": {
        "collection_id": COLLECTION_ID,
    },
    "resource_params": {
        "fragment_mode": "document",
        "fetch_document": True,
        "fetch_navigation": False,
        "add_head_to_content": False,
        "include_breadcrumb": False,
    },
}

### B. Récupération structurée par navigation DTS

In [ ]:
config_navigation = {
    "collection_params": {
        "collection_id": COLLECTION_ID,
    },
    "resource_params": {
        "fragment_mode": "navigation",
        "fetch_document": True,
        "fetch_navigation": True,
        "add_head_to_content": False,
        "include_breadcrumb": True,
        "exclude_heads_contains": COMMON_EXCLUDED_HEADS,
    },
}

### C. Récupération TEI personnalisée par division

In [ ]:
config_tei_div = {
    "collection_params": {
        "collection_id": COLLECTION_ID,
    },
    "resource_params": {
        "fragment_mode": "tei_xpath",
        "fragment_xpath": ".//tei:text/tei:body/tei:div",
        "title_xpath": "./tei:head",
        "remove_fragment_heads": True,
        "add_head_to_content": False,
        "fetch_document": True,
        "fetch_navigation": False,
        "include_breadcrumb": True,
        "generated_id_prefix": "__DOCUMENT__",
        "exclude_heads_contains": COMMON_EXCLUDED_HEADS,
    },
}

### D. Récupération TEI personnalisée par paragraphe

In [ ]:
config_tei_paragraph = {
    "collection_params": {
        "collection_id": COLLECTION_ID,
    },
    "resource_params": {
        "fragment_mode": "tei_xpath",
        "fragment_xpath": ".//tei:text/tei:body/tei:div/tei:p",
        "title_xpath": "./tei:head",
        "remove_fragment_heads": True,
        "add_head_to_content": False,
        "fetch_document": True,
        "fetch_navigation": False,
        "include_breadcrumb": True,
        "generated_id_prefix": "__DOCUMENT__",
    },
}

## 25. Fonction utilitaire : lancer ThunderDots depuis un dictionnaire

In [ ]:
def run_thunderdots_from_config(
    endpoint_dts: str,
    config: dict,
    *,
    output_path: str | None = None,
    cache_csv_path: str | None = None,
    use_cache: bool = True,
    validate: bool = False,
    verbose: bool = True,
):
    td = ThunderDots(
        endpoint_dts=endpoint_dts,
        collection_params=config.get("collection_params"),
        resource_params=config.get("resource_params"),
        output_path=output_path,
        cache_csv_path=cache_csv_path,
        use_cache=use_cache,
        validate=validate,
        verbose=verbose,
    )
    td.fetch()
    return td, td.results()


# td, results = run_thunderdots_from_config(
#     ENDPOINT_DTS,
#     config_tei_div,
#     output_path=OUTPUT_JSON,
#     cache_csv_path=CACHE_CSV,
#     use_cache=True,
#     validate=True,
# )

## 26. Bonnes pratiques

### Choisir la granularité

- Pour une récupération brute : `fragment_mode="document"`.
- Pour respecter une édition ou un système de citation : `fragment_mode="navigation"`.
- Pour une indexation fine et contrôlée : `fragment_mode="tei_xpath"`.

### Gérer les titres

- Mettez `add_head_to_content=True` si le titre doit influencer la recherche.
- Mettez `add_head_to_content=False` si le titre doit rester une métadonnée séparée.
- En TEI XPath, gardez `remove_fragment_heads=True` pour éviter les doublons.

### Gérer les identifiants

- Avec navigation, `dots_id` vient de la navigation DTS.
- Avec TEI XPath, `dots_id` vient du `xml:id` si disponible.
- Si aucun `xml:id` n’est disponible, ThunderDots génère un ID stable avec `generated_id_prefix` + SHA1.

### Gérer les caches

- Pendant le développement, utilisez `use_cache=False` pour tester les changements.
- En production ou lors d’itérations sur l’analyse, utilisez `use_cache=True` et `output_path`.

### Valider

- Activez `validate=True` lors de la mise en place d’un nouveau corpus.
- Inspectez `results["validation"]` avant d’envoyer les données vers un index.

## 27. Exemple complet

La cellule suivante rassemble une configuration réaliste : récupération d’une collection, métadonnées filtrées, fragmentation TEI personnalisée par division, exclusion de sections non pertinentes, validation et écriture d’artefacts.

In [40]:
td_full = ThunderDots(
    endpoint_dts=ENDPOINT_DTS,
    collection_params={
        "collection_id": COLLECTION_ID,
        "metadata_dublincore": ["title"],
    },
    resource_params={
        "metadata_dublincore": ["identifier", "title", "creator", "date"],
        "metadata_extensions": ["dct:coverage"],
        "fragment_mode": "tei_xpath",
        "fragment_xpath": ".//tei:text/tei:body/tei:div",
        "title_xpath": "./tei:head",
        "remove_fragment_heads": True,
        "add_head_to_content": False,
        "fetch_document": True,
        "fetch_navigation": False,
        "include_breadcrumb": True,
        "generated_id_prefix": "__DOCUMENT__",
        "exclude_heads_contains": COMMON_EXCLUDED_HEADS,
    },
    validate=True,
    verbose=True,
    concurrency=6,
    request_timeout=10.0,
    retries=2,
    backoff_ms=300,
    output_path=OUTPUT_JSON,
    cache_csv_path=CACHE_CSV,
    use_cache=True,
)

td_full.fetch()
full_results = td_full.results()
summarize_results(full_results)
pp.pprint(full_results.get("validation"))

Ressources : 24
ID        : ENCPOS_1972_23
Titre     : Quatre paroisses en bas Limousin au xviiie siècle : Allassac, Donzenac, Sadroc, Voutezac, étude économique et sociale
Fragments : 10
dots_id   : r963060
head      : Sources
Aperçu    : Sources Trois dépôts renferment des documents intéressant notre sujet. Les Archives nationales ont livré des plans de routes (série H), des enquêtes révolutionnaires sur la situation économique de la Corrèze (série F14) et la correspondance des intendants avec le Contrôle général (série G7). Aux Arc…
ID        : ENCPOS_1972_21
Titre     : Les forges des princes de Condé en Bretagne aux xviie et xviiie siècles
Fragments : 18
dots_id   : r962117
head      : Sources
Aperçu    : Sources Les principales sources de cette étude sont les comptes des forges de Bretagne conservés au Musée Condé, au château de Chantilly (série F), les registres des procès-verbaux des séances du conseil des princes de Condé de 1669 à 1788, les enquêtes de 1764, 1783 et 1788 (Arc

## 28. Diagnostic rapide

Quelques problèmes fréquents :

| Symptôme | Cause possible | Solution |
|---|---|---|
| `resource_results` vide | mauvais `collection_id` ou endpoint inaccessible | tester `/collection?id=...` dans un navigateur |
| fragments vides | XML inattendu ou XPath incorrect | inspecter le XML retourné par `/document` |
| un seul fragment malgré `navigation` | pas de citation tree ou navigation absente | utiliser `tei_xpath` |
| titres présents dans `content` | `add_head_to_content=True` ou extraction globale | passer `add_head_to_content=False` |
| sections inutiles indexées | titres non filtrés | enrichir `exclude_heads_contains` |
| IDs générés nombreux | nœuds TEI sans `xml:id` | ajouter des `xml:id` ou accepter les SHA1 stables |

## 29. Conclusion

ThunderDots fournit une couche légère entre un endpoint DTS et vos chaînes d’exploitation documentaire.

Le point important est de choisir la bonne unité de travail :

- **ressource entière** avec `document` ;
- **structure publiée** avec `navigation` ;
- **granularité personnalisée** avec `tei_xpath`.

Une fois les résultats produits, ils peuvent être validés, sauvegardés, exportés vers Elasticsearch ou Qdrant, ou transformés en documents plats pour n’importe quel pipeline de recherche, d’annotation, de RAG ou d’analyse de corpus.